# Run the ensemble extractions operationally - on all the data

We've got the transcription models to be good enough - let's put them to use.

In [1]:
# Break the full dataset into batches of 50,000 images arranged for simple transcription
#  we will then do the extractions batch by batch, and then combine the results into a single dataset for analysis.
#
# On SCRATCH - or we'll run out of disc space

import subprocess


# Location of main image dataset, already sized and filtered
SOURCE = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered"
# Location to save sets of images and transcriptions
OUTPUT = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full"
# Output directory on Azure ML datastore
AML_OUTPUT = "operational_full"
# Batch size for partitioning the dataset
BATCH_SIZE = 50000



In [2]:

subprocess.run(
    [
        "python",
        "../../scripts/partition_images_into_batches.py",
        "--source-root",
        SOURCE,
        "--output-root", 
        OUTPUT,
        "--batch-size",
        str(BATCH_SIZE),
    ],
    check=True,
)


Scanning: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered
Total images : 634893
Batch size   : 50000
Batches      : 13

  batch_00: 50000 images (DRain_1861-1870_Alderney-0.jpg … DRain_1891-1900_RainNos_Worcestershire_Wiltshire_B071-8.jpg)
  batch_01: 50000 images (DRain_1891-1900_RainNos_Worcestershire_Wiltshire_B072-0.jpg … DRain_1901-1910_RainNos_Yorkshire_M-Y-640.jpg)
  batch_02: 50000 images (DRain_1901-1910_RainNos_Yorkshire_M-Y-641.jpg … DRain_1911-1920_RainNos_Shropshire_B051-1.jpg)
  batch_03: 50000 images (DRain_1911-1920_RainNos_Shropshire_B051-10.jpg … DRain_1921-1930_RainNos_Hampshire_L-S-433.jpg)
  batch_04: 50000 images (DRain_1921-1930_RainNos_Hampshire_L-S-434.jpg … DRain_1921-1930_RainNos_Yorkshire_H-62.jpg)
  batch_05: 50000 images (DRain_1921-1930_RainNos_Yorkshire_H-63.jpg … DRain_1931-1940_RainNos_1881-1911_B027-1.jpg)
  batch_06: 50000 images (DRain_1931-1940_RainNos_1881-1911_B027-2.jpg … DRain_1931-1940_RainNos_3672-3700_B028-4.jpg)
 

CompletedProcess(args=['python', '../../scripts/partition_images_into_batches.py', '--source-root', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered', '--output-root', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full', '--batch-size', '50000'], returncode=0)

In [27]:
# Everything below is done batch by batch, to make it easier to stop and restart.

# Pick batch, and then run the following cells to do the extractions for that batch. Then repeat for the next batch.
BATCH=11

OUTPUT_B=f"{OUTPUT}/batch_{BATCH:02d}"
AML_OUTPUT_B=f"{AML_OUTPUT}/batch_{BATCH:02d}"

print(f"Batch {BATCH} will be processed from {OUTPUT_B} and saved to {AML_OUTPUT_B}")

Batch 11 will be processed from /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/batch_11 and saved to operational_full/batch_11


In [ ]:
# Upload the sampled images to the Azure ML datastore:
# Deleting the existing contents of the datastore directory first, if necessary
subprocess.run(
    [
        "bash",
        "../../scripts/aml_delete.sh",
        AML_OUTPUT_B,
    ],
    check=True,
)
subprocess.run(
    [
        "bash",
        "../../scripts/aml_upload.sh",
        "--src",
        OUTPUT_B,
        "--dst",
        AML_OUTPUT_B,
    ],
    check=True,
)   

# Note, this cell produces 13Mb of output, delete the outputs a.s.a.p - we don't want to fill git up with them.

That's got the images selected and uploaded. Next step is to run an extraction on those images with each of the operational models.

In [34]:
# Basic setup - model names, batch sizes, etc.
# Specify the model checkpoints manually, rather than auto-generating, so we can be clear what we've used.
# Here I've just specified the 2nd order models.

import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, checkpoint_path, batch_size, total_shards)
# Skip Ministral for now, as it is much slower than the other models
MODEL_SETTINGS = [
    ("SmolVLM", "Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260629-150922/HuggingFaceTB--SmolVLM2-2.2B-Instruct", 50, 1),
    ("Granite4", "Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260629-150950/ibm-granite--granite-vision-4.1-4b", 50, 1),
    ("Gemma3", "Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260629-151007/google--gemma-3-4b-it", 50, 1),
    ("Gemma4", "Daily_rainfall_sample/outputs/checkpoints/google--gemma-4-E4B-it-20260629-151026/google--gemma-4-E4B-it", 50, 1),
    ("Ministral", "Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Small-3.1-24B-Instruct-2503-20260629-151046/mistralai--Mistral-Small-3.1-24B-Instruct-2503", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Node GPU workers per extraction job: 8


In [80]:
# Now run an extraction job, with each checkpoint, on the image sample.

for model_name, checkpoint, batch_size, total_shards in MODEL_SETTINGS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            AML_OUTPUT_B + "/images",
            "--transcriptions-path",
            AML_OUTPUT_B + "/transcriptions",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/operational_full/batch_12/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Smal

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

bold_lemon_k48rqlm0s4
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260713-090039
  Model: smolvlm
  Dataset: operational_full/batch_12/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [28]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = Path(AML_OUTPUT_B + "/images")

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")

def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(str(TARGET_IMAGES_PATH))

for model_name, checkpoint, _batch_size, _total_shards in MODEL_SETTINGS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )


print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")


Ministral: 20260713-085935

Copy this block into the next cell:

RUN_NAMES = [
    "20260713-085935",
]



In [29]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = [  # Batch 11
    "20260713-085935",
]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

Using run names:
   20260713-085935


In [30]:
# When the jobs have completed successfully,
#  download the extractions so we can analyze them locally.
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../../scripts/aml_download.sh", "--run-name", run_name, "--output-dir",
          f"{OUTPUT}/individual_transcriptions/{run_name}"],
        check=True,
    )

Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/individual_transcriptions/20260713-085935/extractions/20260713-085935
         workers: 16


Finished[#############################################################]  100.0000%.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_4092-4121_B015-5.json"[]  100.0000%%


[
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-389.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-39.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-390.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-391.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-392.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260713-085935/DRain_1951-1962_RainNos_2353-2373-393.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Ins

Assemble the 5 different model transcriptions into one ensemble transcriptions output directory.

This will contain 1 file per image, same format as the regular transcriptions, except that for each cell there is an array of 5 transcriptions instead of a single one. If the transcriptions for an individual model are missing (the extraction failed) they are entered as 'missing' in the array.

This is the production output for the transcription process - an ensemble transcription of all pages, ready for QC and further analysis.

In [36]:
subprocess.run(
    [
        "python",
        "../../scripts/build_ensemble_transcriptions.py",
        "--input-dir", f"{OUTPUT}/individual_transcriptions/",
        "--output-dir", f"{OUTPUT}/ensemble_transcriptions/",
    ],
    check=True,
)

Discovering extraction files in ['/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/individual_transcriptions'] ...
Found 584513 stems.
{
  "total_stems": 584513,
  "total_cells": 224452992,
  "fully_present_cells": 220357632,
  "partial_cells": 4095360,
  "empty_cells": 0,
  "parse_failed_or_invalid": 11360,
  "input_dirs": [
    "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/individual_transcriptions"
  ],
  "precision": 3,
  "fully_present_fraction": 0.981754,
  "partial_fraction": 0.018246
}


CompletedProcess(args=['python', '../../scripts/build_ensemble_transcriptions.py', '--input-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/individual_transcriptions/', '--output-dir', '/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/'], returncode=0)